In [14]:
import os
import pandas as pd

def count_lines(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        return sum(1 for _ in f)

train_data = pd.read_csv('Data/drug_review_train.csv')
val_data = pd.read_csv('Data/drug_review_validation.csv')
test_data = pd.read_csv('Data/drug_review_test.csv')

#Define the features (X) and the target variable (y)
label = "rating"
X_train = train_data.drop(columns=[label])  # Drop the target column
y_train = train_data[label]  # Target column
X_val = val_data.drop(columns=[label])  # Drop the target column
y_val = val_data[label]  # Target columnX_train = train_data.drop(columns=[label])  # Drop the target column
X_test = test_data.drop(columns=[label])  # Drop the target column
y_test = test_data[label]  # Target column

#print(train_data.info())
#print(y_train.head())

print(pd.Series.nunique(X_train['drugName']))

2865


In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

# Get the text column (update if your text column has a different name)
text_column = 'review'

# Fill missing values
X_train[text_column] = X_train[text_column].fillna("")
X_val[text_column] = X_val[text_column].fillna("")
X_test[text_column] = X_test[text_column].fillna("")

# Initialize TF-IDF
tfidf = TfidfVectorizer(max_features=5000)  # You can tune this

# Fit TF-IDF on training data and transform all sets
X_train_tfidf = tfidf.fit_transform(X_train[text_column])
X_val_tfidf = tfidf.transform(X_val[text_column])
X_test_tfidf = tfidf.transform(X_test[text_column])

In [29]:
def map_sentiment(rating):
    if rating >= 7:
        return 'positive'
    elif rating >= 4:
        return 'neutral'
    else:
        return 'negative'

y_train_sent = y_train.apply(map_sentiment)
y_val_sent = y_val.apply(map_sentiment)
y_test_sent = y_test.apply(map_sentiment)

# Encode strings into integers
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train_sent)
y_val_enc = le.transform(y_val_sent)
y_test_enc = le.transform(y_test_sent)


In [30]:
import lightgbm as lgb
from sklearn.feature_selection import SelectFromModel

# Train LWGBM
lgb_model = lgb.LGBMClassifier(n_estimators=100)
lgb_model.fit(X_train_tfidf, y_train_enc)

# Select features based on importance
selector = SelectFromModel(lgb_model, prefit=True, threshold='median')  # keep top 50%
X_train_sel = selector.transform(X_train_tfidf)
X_val_sel = selector.transform(X_val_tfidf)
X_test_sel = selector.transform(X_test_tfidf)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.799254 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 567749
[LightGBM] [Info] Number of data points in the train set: 110811, number of used features: 5000
[LightGBM] [Info] Start training from score -1.546495
[LightGBM] [Info] Start training from score -2.099891
[LightGBM] [Info] Start training from score -0.408665


In [31]:
import h2o
from h2o.automl import H2OAutoML
from sklearn.metrics import classification_report
import numpy as np

# Start H2O
h2o.init()

# Convert to H2OFrame
train_h2o = h2o.H2OFrame(pd.DataFrame(X_train_sel.toarray()))
train_h2o['label'] = h2o.H2OFrame(y_train_enc.astype('int'))

val_h2o = h2o.H2OFrame(pd.DataFrame(X_val_sel.toarray()))
val_h2o['label'] = h2o.H2OFrame(y_val_enc.astype('int'))

# Run AutoML
aml = H2OAutoML(max_models=10, seed=1)
aml.train(x=train_h2o.col_names[:-1], y='label', training_frame=train_h2o)

# Evaluate on validation set
preds = aml.predict(val_h2o).as_data_frame()['predict']
print(classification_report(y_val_enc, preds.astype(int), target_names=le.classes_))

# Shut down H2O (optional at the end)
# h2o.shutdown(prompt=False)


ModuleNotFoundError: No module named 'h2o'